In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import ast

# Load Databooks

In [0]:
# 1. Define the base directory generically
DATA_DIR = Path('C:/Users/iamph/OneDrive/Documents/Datasets/Kaggle_Fitness_Workout_Dataset')

# 2. Safely join the path to the file
file_path = DATA_DIR / 'programs_detailed_boostcamp_kaggle_LABELLED_CLEAN_TITLE.csv'

ex = pd.read_csv(file_path)

TODO: Load the original dataset and then add the labels, day labels and clean dataset

In [0]:
df = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_dataset_CLEAN.csv')

In [0]:
ex = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-analysis-datasets/programs_detailed_boostcamp_kaggle.csv')

In [0]:
def norm(s):
    s = str(s).lower()
    s = re.sub(r'\(.*?\)', '', s)          # strip equipment notes in parens
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

In [0]:
lookup = labeled.drop_duplicates('norm').set_index('norm')['label']

unique_names['label'] = unique_names['norm'].map(lookup)

In [0]:
to_predict = pd.read_csv('/Workspace/Users/iamphillipluong@gmail.com/Datasets/exercise-classification/exercises_to_predict_MANUAL_LABELED.csv')

In [0]:
to_predict['final_label'] = to_predict['manual_label']
to_predict['final_label'] = to_predict['final_label'].fillna(to_predict['my_label'])

label_dict = dict(zip(to_predict['exercise_name'], to_predict['final_label']))

In [0]:
unique_names['model_labels'] = unique_names['exercise_name'].map(label_dict)
unique_names['label'] = unique_names['label'].fillna(unique_names['model_labels'])

program_label = dict(zip(unique_names['exercise_name'], unique_names['label']))
prog['exercise_label'] = prog['exercise_name'].map(program_label)

In [0]:
prog[prog.equipment.isna()].title.unique()

In [0]:
prog.drop(prog[prog.title == 'Lera'].index, inplace=True)

# Separate programs with duplicate titles (via description)

In [0]:
dup_titles = prog.groupby('title')['description'].nunique().sort_values(ascending=False)
dup_titles[dup_titles > 1]

In [0]:
duplicate_titles = list(dup_titles[dup_titles > 1].index)

In [0]:
# Create a new DataFrame with the duplicate titles
prog2 = pd.DataFrame()
for title in duplicate_titles:
    desc_list = prog[(prog.title == title)].description.unique()
    tmp_title = 1
    for desc in desc_list:
        tmp = prog[(prog.title == title) & (prog.description == desc)]
        tmp['title'] = title + f' ({tmp_title})'
        prog2 = pd.concat([prog2, tmp])
        tmp_title += 1


In [0]:
# Drop the original duplicate title rows 
prog.drop(prog[prog.title.isin(duplicate_titles)].index, inplace=True)

In [0]:
# Replace with the same rows (with new titles)
prog = pd.concat([prog, prog2])
prog.reset_index(inplace=True)

prog.drop(columns=['index'], inplace=True)

# Clean databook, adjust the level and goal columns

In [0]:
ex.level = ex.level.apply(ast.literal_eval)
ex.goal = ex.goal.apply(ast.literal_eval)

In [0]:
ex.drop(ex[ex.title == 'Mania (Upper/Lower)'].index, inplace = True)
ex.drop(ex[ex.title == 'Viking strong'].index, inplace = True)

In [0]:
level_map = {'Beginner': 0, 'Novice': 1, 'Intermediate': 2, 'Advanced': 3}
level_reverse = {value: key for key, value in level_map.items()}

In [0]:
def combine_goals(group):
    unique_goals = sorted({g for sublist in group for g in sublist})
    return str([str(g) for g in unique_goals])

def combine_levels(group):
    mapped = group.explode().map(level_map)
    lo, hi = int(mapped.min()), int(mapped.max())
    return str([level_reverse[n] for n in range(lo, hi + 1)])

ex['goal'] = ex.groupby('title')['goal'].transform(combine_goals)
ex['level'] = ex.groupby('title')['level'].transform(combine_levels)

# Create a new summary sheet

In [0]:
group_keys = ["title", "week", "day"]

# count how often each exercise_label shows up within each day
counts = ex.groupby(group_keys + ["exercise_label"]).size().reset_index(name="n")

max_n = counts.groupby(group_keys)["n"].transform("max")
top_labels = counts[counts["n"] == max_n]

day_label = (
    top_labels.sort_values("exercise_label")
    .groupby(group_keys)["exercise_label"]
    .apply(lambda s: ", ".join(s))   # e.g. "Back, Legs" on ties
    .reset_index(name="day_label")
)

ex = ex.merge(day_label, on=group_keys, how="left")

In [0]:
ex.day_label = ex.day_label.apply(ast.literal_eval)

# Program Summary Sheet (include the workout days)

In [0]:
titles = ex.title.unique()

records = []
groups = dict(tuple(ex.groupby('title')))

for title in titles:
    tmp = groups[title]

    desc_notna = tmp.description.dropna()
    description = desc_notna.iloc[0] if not desc_notna.empty else 'N/A'

    tmp
    level_set = list(set().union(*tmp.level.dropna().values))
    goal_set = list(set().union(*tmp.goal.dropna().values))
    day_set = list(tmp.day_label.unique())

    last_week = int(tmp.week.max())

    day_max_per_week = tmp.groupby('week')['day'].max()
    intensity_per_day = tmp.groupby(['week', 'day'])['intensity'].mean()

    row = {
        'title': title,
        'description': description,
        'level': str(level_set),
        'goal': str(goal_set),
        'equipment': tmp.equipment.values[0],
        'program_length_weeks': last_week,
        'time_per_workout': tmp.time_per_workout.values[0],
        'total_exercises': len(tmp),
        'program_length_days': int(day_max_per_week.loc[1:last_week].sum()),
        'average_intensity': round(intensity_per_day.mean(), 2),
        'workout_days': str(day_set),
    }
    records.append(row)

df3 = pd.DataFrame(records)

In [0]:
df3.to_csv(DATA_DIR / 'program_summary_v3_1.csv')